# 03 — Compute and store LOT embeddings in Stage 2

This notebook computes multiple reference/solver/representation variants and saves each one under `lot_embeddings/{preprocess_id}/{embedding_id}` in the existing Stage 2 HDF5 file. It also verifies shapes, convergence, finite values, transport costs, and reproducibility metadata.

In [ ]:
from pathlib import Path
import json
import time
import h5py
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from flowlot.evaluation.repeated_benchmark import load_registry
from flowlot.io import Stage2Loader, audit_stage2
from flowlot.transport import compute_stage2_embeddings

## Experiment configuration

Available references are `patient0`, `pooled`, `gaussian`, `uniform`, and `barycenter`. Solvers are `sinkhorn`, `emd`, `linprog`, and equal-size-only `hungarian`. Every experiment needs a unique ID so variants cannot overwrite one another.

In [ ]:
STAGE2 = Path('../data/stage2_analytics.h5')
DATASET, CELL_COUNT, PREPROCESS = 'BLAST110', '1000', 'common12'
RANDOM_STATE = 42
RUN_EMBEDDINGS = False  # Safety switch after preflight checks.
STOP_ON_ERROR = True

EXPERIMENTS = [
    {
        'id': 'patient0_sinkhorn_disp', 'reference': 'patient0', 'solver': 'sinkhorn',
        'reference_size': 512, 'representation': 'displacement',
        'reference_kwargs': {'index': 0}, 'solver_kwargs': {'reg': 0.01, 'max_iter': 10000},
        'store_transport': False,
    },
    {
        'id': 'pooled_sinkhorn_disp', 'reference': 'pooled', 'solver': 'sinkhorn',
        'reference_size': 512, 'representation': 'displacement',
        'reference_kwargs': {}, 'solver_kwargs': {'reg': 0.01, 'max_iter': 10000},
        'store_transport': False,
    },
    {
        'id': 'gaussian_sinkhorn_disp', 'reference': 'gaussian', 'solver': 'sinkhorn',
        'reference_size': 512, 'representation': 'displacement',
        'reference_kwargs': {'jitter': 1e-6}, 'solver_kwargs': {'reg': 0.01},
        'store_transport': False,
    },
    # Barycenter and exact EMD/linprog are substantially slower; enable deliberately.
    # {'id': 'barycenter_emd_disp', 'reference': 'barycenter', 'solver': 'emd',
    #  'reference_size': 256, 'representation': 'displacement',
    #  'reference_kwargs': {'max_patients': 10, 'max_cells_per_patient': 1024, 'max_iter': 50},
    #  'solver_kwargs': {}, 'store_transport': False},
]

## Reference cohort and leakage control

For exploratory embeddings, `REFERENCE_PATIENT_IDS=None` uses all available patients. For confirmatory classification, build each reference from training patients only. The optional registry code selects one run/k cohort; use a unique embedding ID that includes that run/k.

In [ ]:
REFERENCE_PATIENT_IDS = None
SPLIT_REGISTRY = Path('../results/repeated_classification/shared_splits.json')
USE_TRAINING_REFERENCE = False
REFERENCE_RUN, REFERENCE_K = 0, 8
if USE_TRAINING_REFERENCE:
    registry = load_registry(SPLIT_REGISTRY)
    REFERENCE_PATIENT_IDS = registry['splits'][REFERENCE_RUN]['train_ids_by_k'][str(REFERENCE_K)]
    print(f'Using {len(REFERENCE_PATIENT_IDS)} training patients for the reference')
else:
    print('Exploratory mode: references may use the full cohort.')

## Stage 2 and computational preflight

OT memory scales approximately with `target_cells × reference_cells`. Coupling matrices are the largest optional output; leave `store_transport=False` unless individual transport plans are needed.

In [ ]:
inventories, initial_issues = audit_stage2(STAGE2)
display(initial_issues)
assert initial_issues.empty, 'Correct Stage 2 errors before computing LOT'
processed = inventories['preprocess'].query(
    'dataset == @DATASET and cell_count == @CELL_COUNT and preprocess == @PREPROCESS'
)
assert not processed.empty, 'Requested preprocess group was not found'
preflight_rows = []
for experiment in EXPERIMENTS:
    reference_cells = experiment['reference_size']
    coupling_entries = int((processed['n_cells'] * reference_cells).sum())
    preflight_rows.append({
        'embedding': experiment['id'], 'patients': processed['patient_id'].nunique(),
        'tubes': processed['tube'].nunique(), 'reference_cells': reference_cells,
        'estimated_coupling_GB_float64': coupling_entries * 8 / 1e9,
        'store_transport': experiment['store_transport'],
    })
preflight = pd.DataFrame(preflight_rows)
display(preflight)
assert preflight['embedding'].is_unique, 'Experiment IDs must be unique'
assert all('/' not in value and value.strip() for value in preflight['embedding']), 'IDs must be non-empty HDF5 keys without slashes'
for experiment in EXPERIMENTS:
    if experiment['solver'] == 'hungarian':
        assert (processed['n_cells'] == experiment['reference_size']).all(), 'Hungarian requires every target and reference to have equal cell counts'

## Compute and save embeddings

Each successful experiment writes its reference matrix, patient IDs, reference-patient IDs, flattened embeddings, sorted point clouds, costs, convergence flags, optional transport plans, and JSON-encoded parameters directly into Stage 2.

In [ ]:
run_records = []
if RUN_EMBEDDINGS:
    for experiment in EXPERIMENTS:
        started = time.perf_counter()
        try:
            shapes = compute_stage2_embeddings(
                STAGE2, DATASET, CELL_COUNT, PREPROCESS,
                reference_type=experiment['reference'], solver=experiment['solver'],
                reference_size=experiment['reference_size'],
                representation=experiment['representation'], random_state=RANDOM_STATE,
                store_transport=experiment['store_transport'], overwrite=True,
                reference_patient_ids=REFERENCE_PATIENT_IDS,
                reference_kwargs=experiment['reference_kwargs'],
                solver_kwargs=experiment['solver_kwargs'], embedding_id=experiment['id'],
            )
            run_records.append({'embedding': experiment['id'], 'status': 'ok', 'seconds': time.perf_counter() - started, 'shapes': json.dumps(shapes)})
        except Exception as error:
            run_records.append({'embedding': experiment['id'], 'status': 'error', 'seconds': time.perf_counter() - started, 'message': repr(error)})
            if STOP_ON_ERROR:
                raise
    display(pd.DataFrame(run_records))
else:
    print('Dry run. Review preflight, then set RUN_EMBEDDINGS=True.')

## Verify saved embeddings

The audit checks patient alignment, flattened width, finite values, reference shape, cost/convergence lengths, and sorted-point-cloud coverage.

In [ ]:
inventories, final_issues = audit_stage2(STAGE2)
embedding_inventory = inventories['embeddings']
requested_ids = {experiment['id'] for experiment in EXPERIMENTS}
saved = embedding_inventory.query(
    'dataset == @DATASET and cell_count == @CELL_COUNT and preprocess == @PREPROCESS and embedding in @requested_ids'
).sort_values(['embedding', 'tube'])
display(saved)
display(final_issues)
if RUN_EMBEDDINGS:
    assert set(saved['embedding']) == requested_ids, 'One or more embedding experiments are missing'
    assert final_issues.empty, 'Saved embedding verification failed'

In [ ]:
if not saved.empty:
    with h5py.File(STAGE2) as handle:
        for _, row in saved.iterrows():
            path = f"{DATASET}/{CELL_COUNT}/{row['tube']}/lot_embeddings/{PREPROCESS}/{row['embedding']}"
            group = handle[path]
            print('/' + path)
            print('  attrs:', dict(group.attrs))
            print('  datasets:', sorted(group.keys()))

## Compare saved LOT spaces

PCA below is diagnostic only; the classifier consumes the full stored LOT vectors. Separate panels are necessary because reference choices define different coordinate systems.

In [ ]:
if not saved.empty:
    with Stage2Loader(STAGE2) as loader:
        for embedding_id in sorted(requested_ids & set(saved['embedding'])):
            panels = []
            for tube in sorted(saved.loc[saved['embedding'] == embedding_id, 'tube']):
                patient_ids, vectors = loader.embeddings(DATASET, CELL_COUNT, tube, PREPROCESS, embedding_id)
                metadata = loader.metadata(DATASET, CELL_COUNT, tube)
                label_map = dict(zip(metadata['patient_ids'], metadata['labels']))
                coordinates = PCA(2, random_state=RANDOM_STATE).fit_transform(StandardScaler().fit_transform(vectors))
                panels.append(pd.DataFrame({'PC1': coordinates[:, 0], 'PC2': coordinates[:, 1], 'patient_id': patient_ids, 'label': [label_map[p] for p in patient_ids], 'tube': tube}))
            plot_data = pd.concat(panels, ignore_index=True)
            grid = sns.relplot(data=plot_data, x='PC1', y='PC2', hue='label', col='tube', kind='scatter', facet_kws={'sharex': False, 'sharey': False})
            grid.figure.suptitle(embedding_id, y=1.03)
            plt.show()